# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering

This notebook defines and verifies the data contract used by the downstream content-archetype clustering workflow. It is intentionally self-contained and uses repository-local data only.

## 1. Unit of analysis + time window

**Unit of analysis:** one row represents one content item in the available analysis snapshot.

**Performance window:** the downstream model uses observed 90-day search/performance aggregates (`impressions_90d`, `ctr_90d`, `avg_position_90d`, and `engagement_rate`). Content age and days since update are measured at the snapshot.

This is a descriptive clustering dataset; the time window is not a future outcome label.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_CANDIDATES = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../outputs/content_archetypes_clustered.parquet"),
    Path("../outputs/content_level_model_dataset.parquet"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No supported data file found. Add the anonymized CSV under data/raw/ or run W05 first.")

df = pd.read_csv(DATA_PATH) if DATA_PATH.suffix.lower() == ".csv" else pd.read_parquet(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Column count:", len(df.columns))

id_col = "content_id" if "content_id" in df.columns else "content_hash_id"
assert id_col in df.columns
assert len(df) > 0
print("Unit-of-analysis dataset loaded: PASS")

## 2. Fields: feature / label / context / excluded

This project is **unsupervised clustering**, so there is no prediction label. The eight fields below are the core numeric features used by W05. Identifiers are context only. Query breadth and future/trend fields are excluded from the clustering vector.

In [ ]:
core_features = [
    "search_volume", "word_count",
    "content_age_days", "days_since_update",
    "impressions_90d", "ctr_90d",
    "avg_position_90d", "engagement_rate",
]

field_contract = pd.DataFrame([
    ("search_volume", "feature", "Observed topic search-demand signal", "median", "snapshot"),
    ("word_count", "feature", "Content length in words", "median", "snapshot"),
    ("content_age_days", "feature", "Content age at snapshot", "median", "snapshot"),
    ("days_since_update", "feature", "Elapsed days since recorded update", "median", "snapshot"),
    ("impressions_90d", "feature", "Observed search impressions in 90-day window", "median + log1p", "90-day window"),
    ("ctr_90d", "feature", "Observed click-through rate in 90-day window", "median", "90-day window"),
    ("avg_position_90d", "feature", "Observed average position; zero means no position data", "0→NaN, then median", "90-day window"),
    ("engagement_rate", "feature", "Observed engagement rate", "median", "analysis window"),
    ("client_id / client_hash_id", "context", "Join/group identifier only", "not a feature", "snapshot"),
    ("content_id / content_hash_id", "context", "Content identifier only", "not a feature", "snapshot"),
    ("cluster", "output", "W05-generated cluster assignment", "not a source feature", "after modeling"),
    ("query breadth", "excluded", "May encode data coverage rather than content archetype", "excluded", "90-day window"),
    ("future / trend / label fields", "excluded", "Could introduce later information", "excluded", "future"),
], columns=["field","bucket","meaning","missing_treatment","availability"])
display(field_contract)

## 3. Verify it with queries (grain, counts, missing values, windows)

The checks below verify one-row-per-content grain, missingness, required fields, and any date columns that are actually available in the source. No date range is invented when the source does not contain explicit window dates.

In [ ]:
required = [id_col] + core_features
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required contract fields: {missing_required}")

client_col = "client_id" if "client_id" in df.columns else ("client_hash_id" if "client_hash_id" in df.columns else None)
print("Rows:", len(df))
print("Unique content IDs:", df[id_col].nunique())
print("Duplicate content IDs:", int(df[id_col].duplicated().sum()))
if client_col:
    print("Unique clients:", df[client_col].nunique())

missing_core = df[core_features].isna().sum().to_frame("missing_n")
missing_core["missing_pct"] = (missing_core["missing_n"] / len(df) * 100).round(2)
display(missing_core.sort_values("missing_n", ascending=False))

for col in ["report_date","window_start","window_end"]:
    if col in df.columns:
        s = pd.to_datetime(df[col], errors="coerce")
        print(f"{col}: {s.min()} to {s.max()}")

if "avg_position_90d" in df.columns:
    print("avg_position_90d zero rows:", int((df["avg_position_90d"] == 0).sum()))

assert df[id_col].nunique() == len(df), "Contract requires one row per content item."
print("Grain check: PASS")

## 4. Data limits

This contract supports **descriptive clustering and decision support only**. The data can show observed differences among content items, but it cannot establish that changing content causes later traffic, ranking, CTR, or engagement outcomes.

Important limitations: 90-day performance aggregates are a snapshot; missing values and median imputation can influence cluster boundaries; grouped-client validation tests unseen-client generalization rather than future-time generalization; rare clusters may not represent the whole portfolio; and the structured dataset does not provide semantic article understanding.

For W05, `avg_position_90d = 0` is treated as missing rather than a genuine rank.

In [ ]:
limits = [
    "No causal inference from cluster membership",
    "90-day observations do not represent every content lifecycle",
    "Missingness and median imputation can influence cluster boundaries",
    "Unseen-client validation is not future-time validation",
    "Rare clusters should be treated as narrow observed patterns",
    "The model is metric-based and does not infer article semantics",
]
for i, item in enumerate(limits, 1):
    print(f"{i}. {item}")

assert len(core_features) == 8
assert all(c in df.columns for c in core_features)
assert df[id_col].nunique() == len(df)
print("Data-contract assertions: PASS")

## Self-check

Run the notebook top-to-bottom on a fresh Colab/Jupyter runtime. The checks below cover the mechanical contract requirements.

In [ ]:
checks = [
    ("Data source loaded", len(df) > 0),
    ("One row per content", df[id_col].nunique() == len(df)),
    ("Exactly eight core features", len(core_features) == 8),
    ("All core features available", all(c in df.columns for c in core_features)),
    ("No identifier in core vector", not any(c in core_features for c in ["content_id","content_hash_id","client_id","client_hash_id"])),
    ("No future/label-like field in core vector", not any(k in c.lower() for c in core_features for k in ["future","trend","label","target","outcome"])),
    ("No query field in core vector", not any("query" in c.lower() for c in core_features)),
]
check_df = pd.DataFrame(checks, columns=["check","passed"])
display(check_df)
if not check_df["passed"].all():
    raise AssertionError("One or more W03 data-contract checks failed.")
print("All W03 data-contract checks PASS.")